## Red de tres neuronas para aprender XOR

XOR devuelve $1$ cuando sus entradas son distintas y $0$ cuando son iguales:

$$
\begin{array}{cc|c}
x_1 & x_2 & y \\
\hline
0 & 0 & 0 \\
0 & 1 & 1 \\
1 & 0 & 1 \\
1 & 1 & 0
\end{array}
$$

La red tiene dos neuronas ocultas y una neurona de salida. Las entradas no se cuentan como neuronas de cálculo:

$$
(x_1, x_2)
\longrightarrow
(a_1, a_2)
\longrightarrow
\hat{y}
$$

### Propagación hacia delante

Cada neurona oculta $j$ calcula una suma ponderada de las entradas:

$$
z_j = x_1 w_{j1} + x_2 w_{j2} + b_j
$$

y aplica la función sigmoide:

$$
a_j = \sigma(z_j) = \frac{1}{1 + e^{-z_j}}
$$

Aquí, $j \in \{1,2\}$ identifica una de las dos neuronas ocultas. Cada neurona tiene sus propios pesos $w_{j1}, w_{j2}$ y su sesgo $b_j$.

Las salidas de esas dos neuronas se conectan con la neurona final:

$$
z_o = a_1 v_1 + a_2 v_2 + b_o
$$

$$
\hat{y} = \sigma(z_o)
$$

Los pesos $v_1$ y $v_2$ controlan cuánto influye cada neurona oculta en la salida final.

### Error y ajuste de la neurona de salida

Usamos el error cuadrático:

$$
L = \frac{1}{2}(y - \hat{y})^2
$$

La sigmoide tiene esta derivada:

$$
\sigma'(z) = \sigma(z)(1 - \sigma(z))
$$

Definimos el ajuste de la neurona de salida como:

$$
\delta_o = (y - \hat{y})\hat{y}(1 - \hat{y})
$$

Con tasa de aprendizaje $\eta$, actualizamos sus pesos y su sesgo:

$$
v_j \leftarrow v_j + \eta \delta_o a_j
$$

$$
b_o \leftarrow b_o + \eta \delta_o
$$

### Propagación del error hacia las neuronas ocultas

La salida final depende de las neuronas ocultas, así que el error también permite ajustar sus pesos. Para cada neurona oculta $j$:

$$
\delta_j = \delta_o v_j a_j(1 - a_j)
$$

Luego actualizamos los pesos que conectan las entradas con esa neurona:

$$
w_{ji} \leftarrow w_{ji} + \eta \delta_j x_i
$$

y su sesgo:

$$
b_j \leftarrow b_j + \eta \delta_j
$$

Así, el error de la predicción final se propaga hacia atrás por las conexiones. La red repite este proceso con los ejemplos para ir ajustando todos sus pesos y sesgos.

In [1]:
import math
import random


def sigmoide(x):
    return 1 / (1 + math.exp(-x))


# Entradas y respuestas correctas para XOR
datos = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]

# Inicializamos con valores pequeños y distintos
generador = random.Random(7)

# Dos neuronas ocultas; cada una tiene un peso por entrada
pesos_ocultos = [
    [generador.uniform(-1, 1), generador.uniform(-1, 1)],
    [generador.uniform(-1, 1), generador.uniform(-1, 1)],
]
sesgos_ocultos = [
    generador.uniform(-1, 1),
    generador.uniform(-1, 1),
]

# La neurona de salida tiene un peso por neurona oculta
pesos_salida = [
    generador.uniform(-1, 1),
    generador.uniform(-1, 1),
]
sesgo_salida = generador.uniform(-1, 1)

tasa_aprendizaje = 0.5
epocas = 30_000


# Entrenamiento
for epoca in range(epocas):
    for entradas, esperado in datos:
        x1, x2 = entradas

        # Propagación hacia delante: capa oculta
        salidas_ocultas = []

        for j in range(2):
            suma = (
                x1 * pesos_ocultos[j][0]
                + x2 * pesos_ocultos[j][1]
                + sesgos_ocultos[j]
            )
            salidas_ocultas.append(sigmoide(suma))

        # Propagación hacia delante: neurona de salida
        suma_salida = (
            salidas_ocultas[0] * pesos_salida[0]
            + salidas_ocultas[1] * pesos_salida[1]
            + sesgo_salida
        )
        prediccion = sigmoide(suma_salida)

        # Ajuste de la neurona de salida
        delta_salida = (
            (esperado - prediccion)
            * prediccion
            * (1 - prediccion)
        )

        # Guardamos los pesos actuales para propagar el error hacia atrás
        pesos_salida_anteriores = pesos_salida.copy()

        # Ajuste de cada neurona oculta
        deltas_ocultos = []

        for j in range(2):
            salida_oculta = salidas_ocultas[j]

            delta_oculta = (
                delta_salida
                * pesos_salida_anteriores[j]
                * salida_oculta
                * (1 - salida_oculta)
            )
            deltas_ocultos.append(delta_oculta)

        # Actualizamos las conexiones hacia la neurona de salida
        for j in range(2):
            pesos_salida[j] += (
                tasa_aprendizaje
                * delta_salida
                * salidas_ocultas[j]
            )

        sesgo_salida += tasa_aprendizaje * delta_salida

        # Actualizamos las conexiones desde las entradas
        for j in range(2):
            pesos_ocultos[j][0] += (
                tasa_aprendizaje * deltas_ocultos[j] * x1
            )
            pesos_ocultos[j][1] += (
                tasa_aprendizaje * deltas_ocultos[j] * x2
            )
            sesgos_ocultos[j] += (
                tasa_aprendizaje * deltas_ocultos[j]
            )


# Comprobamos las predicciones aprendidas
for entradas, esperado in datos:
    x1, x2 = entradas

    salidas_ocultas = []

    for j in range(2):
        suma = (
            x1 * pesos_ocultos[j][0]
            + x2 * pesos_ocultos[j][1]
            + sesgos_ocultos[j]
        )
        salidas_ocultas.append(sigmoide(suma))

    suma_salida = (
        salidas_ocultas[0] * pesos_salida[0]
        + salidas_ocultas[1] * pesos_salida[1]
        + sesgo_salida
    )
    prediccion = sigmoide(suma_salida)

    clase_predicha = 1 if prediccion >= 0.5 else 0

    print(
        f"{entradas} -> {prediccion:.3f} "
        f"(clase: {clase_predicha}, esperado: {esperado})"
    )

[0, 0] -> 0.008 (clase: 0, esperado: 0)
[0, 1] -> 0.492 (clase: 0, esperado: 1)
[1, 0] -> 0.990 (clase: 1, esperado: 1)
[1, 1] -> 0.492 (clase: 0, esperado: 0)


## Cómo se conectan muchas neuronas

Una red neuronal se organiza normalmente en capas:

$$
\text{capa de entrada}
\longrightarrow
\text{capas ocultas}
\longrightarrow
\text{capa de salida}
$$

La capa de entrada contiene los datos. Las neuronas de las capas ocultas transforman esos datos, y la capa de salida produce la predicción.

### Conexión entre dos capas

Supongamos que la capa anterior tiene $n_{l-1}$ valores y la capa actual tiene $n_l$ neuronas. En una capa completamente conectada, cada neurona actual recibe una conexión desde cada valor de la capa anterior.

Para la neurona $j$ de la capa $l$:

$$
z_j^{(l)}
=
\sum_{i=1}^{n_{l-1}}
w_{ji}^{(l)} a_i^{(l-1)}
+
b_j^{(l)}
$$

$$
a_j^{(l)} = \sigma\left(z_j^{(l)}\right)
$$

Aquí:

- $a_i^{(l-1)}$ es la salida de la neurona $i$ de la capa anterior.
- $w_{ji}^{(l)}$ es el peso de la conexión desde esa neurona hasta la neurona $j$ actual.
- $b_j^{(l)}$ es el sesgo de la neurona actual.
- $a_j^{(l)}$ es la salida de la neurona actual.

La misma operación se puede escribir de forma compacta con matrices:

$$
\mathbf{z}^{(l)}
=
\mathbf{W}^{(l)}\mathbf{a}^{(l-1)}
+
\mathbf{b}^{(l)}
$$

$$
\mathbf{a}^{(l)}
=
\sigma\left(\mathbf{z}^{(l)}\right)
$$

La matriz $\mathbf{W}^{(l)}$ guarda todos los pesos que conectan la capa anterior con la actual. Cada fila contiene los pesos que llegan a una neurona.

### Cuántos parámetros hay

Si una capa anterior tiene $n_{l-1}$ neuronas y la siguiente tiene $n_l$, una conexión completa entre ellas tiene:

$$
n_{l-1}n_l
$$

pesos y $n_l$ sesgos.

Por ejemplo, en la red XOR del código hay dos entradas, dos neuronas ocultas y una neurona de salida:

$$
2 \longrightarrow 2 \longrightarrow 1
$$

Las entradas se conectan con las neuronas ocultas mediante $2 \cdot 2 = 4$ pesos. Las dos neuronas ocultas se conectan con la salida mediante $2 \cdot 1 = 2$ pesos. En total hay seis pesos y tres sesgos:

$$
(2 \cdot 2) + (2 \cdot 1) + 2 + 1 = 9
$$

parámetros entrenables.

### Por qué usar varias neuronas

Cada neurona calcula una transformación de sus entradas. Con varias neuronas, la red puede aprender varias transformaciones y combinarlas.

En XOR, ninguna entrada por sí sola determina la respuesta. Las dos neuronas ocultas pueden aprender señales intermedias distintas; la neurona final combina esas señales para producir la salida. No es necesario que cada neurona termine representando una regla lógica exacta: sus pesos se ajustan conjuntamente para reducir el error de la red.

Las capas permiten componer transformaciones:

$$
\mathbf{x}
\longrightarrow
\mathbf{a}^{(1)}
\longrightarrow
\mathbf{a}^{(2)}
\longrightarrow
\hat{\mathbf{y}}
$$

Una capa puede construir representaciones intermedias que facilitan el trabajo de la siguiente.

### Por qué hace falta una función de activación

La sigmoide introduce una transformación no lineal. Si se quitaran las funciones de activación y solo se usaran sumas ponderadas, varias capas seguidas equivaldrían a una sola transformación lineal:

$$
\mathbf{W}^{(2)}\left(\mathbf{W}^{(1)}\mathbf{x}\right)
=
\left(\mathbf{W}^{(2)}\mathbf{W}^{(1)}\right)\mathbf{x}
$$

En ese caso, añadir capas no permitiría aprender relaciones más complejas. Las funciones de activación hacen posible que la red represente patrones no lineales, como XOR.

### Cómo se entrena una red con varias capas

Primero se calcula la predicción capa por capa. Después se mide el error de la salida. La retropropagación distribuye información sobre ese error hacia las capas anteriores usando la regla de la cadena.

Con la convención de ajuste usada en el código —se suma una corrección basada en $y-\hat{y}$—, para una red con sigmoide y error cuadrático:

$$
\boldsymbol{\delta}^{(L)}
=
(\mathbf{y}-\mathbf{a}^{(L)})
\odot
\sigma'\left(\mathbf{z}^{(L)}\right)
$$

$$
\boldsymbol{\delta}^{(l)}
=
\left((\mathbf{W}^{(l+1)})^T
\boldsymbol{\delta}^{(l+1)}\right)
\odot
\sigma'\left(\mathbf{z}^{(l)}\right)
$$

Aquí, $L$ es la última capa y $\odot$ significa multiplicar elemento por elemento. Luego se ajustan pesos y sesgos:

$$
\mathbf{W}^{(l)}
\leftarrow
\mathbf{W}^{(l)}
+
\eta\,
\boldsymbol{\delta}^{(l)}
(\mathbf{a}^{(l-1)})^T
$$

$$
\mathbf{b}^{(l)}
\leftarrow
\mathbf{b}^{(l)}
+
\eta\boldsymbol{\delta}^{(l)}
$$

La tasa de aprendizaje $\eta$ controla el tamaño de los cambios. En cada paso, las conexiones se ajustan para que la predicción se acerque a la respuesta correcta.

### Aspectos prácticos

- **Más neuronas** permiten representar patrones más variados, pero aumentan los cálculos y pueden hacer que la red aprenda demasiado los ejemplos de entrenamiento.
- **Más capas** permiten construir transformaciones sucesivas, aunque pueden hacer más difícil el entrenamiento.
- **Los pesos iniciales suelen ser pequeños y distintos.** Si dos neuronas comienzan con exactamente los mismos pesos, pueden recibir los mismos ajustes y aprender lo mismo.
- **La tasa de aprendizaje importa.** Si es muy alta, los ajustes pueden saltarse una buena solución; si es muy baja, el entrenamiento puede avanzar lentamente.
- **Las épocas** indican cuántas veces se recorre el conjunto de entrenamiento. Muchas épocas no garantizan que la red generalice bien a datos nuevos.
- **La función de pérdida** mide qué tan malas son las predicciones. El ejemplo usa error cuadrático para que la derivación sea fácil de seguir; otras tareas pueden usar otras pérdidas.
- **Una conexión completa no es obligatoria.** Algunas redes conectan solo ciertas neuronas, o usan arquitecturas especializadas para imágenes, secuencias u otros tipos de datos.

La arquitectura, la función de activación, la pérdida, la inicialización y la tasa de aprendizaje trabajan juntas. Cambiar una de ellas puede cambiar tanto la capacidad de la red como la forma en que aprende.